# Module 2: 實際應用 — 人數計算、文字辨識、交通監控

## 學習目標
- 人數即時計算 + 時間序列追蹤
- 區域計數（模擬「入口 / 出口」監控）
- OCR 文字辨識（新聞台跑馬燈）
- 交通工具分類統計

```
本 Module 的應用地圖：

┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│  Application │     │  Application │     │  Application │
│    A: 人數   │     │    B: 文字   │     │    C: 交通   │
│    計算      │     │    辨識      │     │    監控      │
│              │     │              │     │              │
│ ● 即時計數   │     │ ● OCR 辨識  │     │ ● 車種分類   │
│ ● 時間趨勢   │     │ ● 跑馬燈擷取│     │ ● 數量統計   │
│ ● 區域監控   │     │ ● 多語言    │     │ ● 視覺化     │
└──────────────┘     └──────────────┘     └──────────────┘
```

---
## 環境設定（每次開新 Runtime 都要跑一次）

In [ ]:
!pip install ultralytics opencv-python-headless pillow easyocr -q

import cv2
import numpy as np
from PIL import Image
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import time
from collections import Counter
from ultralytics import YOLO

plt.rcParams['font.size'] = 12
# 中文字型（讓 matplotlib 標題顯示中文，不出現方框）
import subprocess as _sp, glob as _g, matplotlib.font_manager as _fm
_sp.run('apt-get -qq install -y fonts-noto-cjk'.split(), capture_output=True)
_cjk = _g.glob('/usr/share/fonts/**/NotoSansCJK*.*c', recursive=True)
if _cjk:
    _fm.fontManager.addfont(_cjk[0]); plt.rcParams['font.family'] = _fm.FontProperties(fname=_cjk[0]).get_name()
plt.rcParams['axes.unicode_minus'] = False

def show_frame(frame, title='', size=(640, 360)):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb).resize(size)
    if title:
        print(title)
    display(img)

def grab_frame(source, timeout=20):
    import subprocess, os
    url = source
    # 'search:關鍵字' → 用 yt-dlp 找「現在正在直播」的串流
    if isinstance(source, str) and source.startswith('search:'):
        try:
            url = subprocess.run(['yt-dlp', 'ytsearch15:' + source[len('search:'):],
                '--match-filter', 'is_live', '--print', '%(webpage_url)s', '-I', '1:1', '--no-warnings'],
                capture_output=True, text=True, timeout=120).stdout.strip().split('\n')[0]
        except Exception:
            url = ''
    # YouTube → 解析直接串流網址
    if url and ('youtube.com' in url or 'youtu.be' in url):
        try:
            url = subprocess.run(['yt-dlp', '-f', 'best[ext=mp4]/best', '-g', url],
                capture_output=True, text=True, timeout=90).stdout.strip().split('\n')[0]
        except Exception:
            url = ''
    # 國道 MJPEG → curl 取一張 JPEG
    if url and 'abs2mjpg' in url:
        data = subprocess.run(['curl', '-s', '--max-time', '8', url], capture_output=True).stdout
        s = data.find(b'\xff\xd8'); e = data.find(b'\xff\xd9', s)
        if s >= 0 and e >= 0:
            return cv2.imdecode(np.frombuffer(data[s:e + 2], np.uint8), cv2.IMREAD_COLOR)
    elif url:
        # 直播/HLS：用 ffmpeg 抓一張幀（比 cv2 穩定）
        tmp = '/tmp/_grab.jpg'
        try:
            subprocess.run(['ffmpeg', '-y', '-loglevel', 'quiet', '-i', url,
                '-frames:v', '1', '-q:v', '3', tmp], timeout=45)
            if os.path.exists(tmp) and os.path.getsize(tmp) > 1500:
                fr = cv2.imread(tmp)
                if fr is not None:
                    return fr
        except Exception:
            pass
        cap = cv2.VideoCapture(url); cap.set(cv2.CAP_PROP_OPEN_TIMEOUT_MSEC, timeout * 1000)
        ret, frame = cap.read(); cap.release()
        if ret:
            return frame
    # 備援：台灣國道即時影像
    print('來源連線失敗，改用國道即時影像備援')
    data = subprocess.run(['curl', '-s', '--max-time', '8',
        'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=10001'], capture_output=True).stdout
    s = data.find(b'\xff\xd8'); e = data.find(b'\xff\xd9', s)
    if s >= 0 and e >= 0:
        return cv2.imdecode(np.frombuffer(data[s:e + 2], np.uint8), cv2.IMREAD_COLOR)
    raise ConnectionError('來源與備援皆失敗，請稍後再試')

STREAMS = {
    # === 交通（台灣國道即時影像，政府 CCTV，最穩定）===
    '國道1號 高架(車多)': 'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=11470',
    '國道1號 基隆端': 'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=10001',
    '國道1號 0K+100': 'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=10010',
    '國道3號': 'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=30001',
    '國道2號 機場': 'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=20000',
    # === 多樣場景（YouTube 直播，search: 自動找「現在正在直播」的，不怕單一 ID 失效）===
    '動物 (非洲 safari 直播)': 'search:african safari waterhole live animals',
    '城市人潮 (路口直播)': 'search:shibuya scramble crossing live',
    '港口/船隻 (直播)': 'search:live harbour port ships cam',
    '野生鳥類 (直播)': 'search:cornell bird feeder live cam',
}

model = YOLO('yolov8n.pt')
print('環境載入完成！')

---
## Application A: 人數即時計算

### 應用場景
- 商場人流監控
- 活動現場即時人數
- 辦公室佔用率分析

```
攝影機畫面
┌────────────────────────┐
│  👤        👤    👤   │
│     👤                 │  → AI 計數: 5 人
│           👤           │  → 信心度 > 50%
│  People: 5             │
└────────────────────────┘
```

### A-1: 單張人數計算

In [ ]:
def count_people(frame, model, conf=0.5):
    """偵測畫面中的人數，回傳人數和標註影像"""
    # classes=[0] 代表只偵測 "person" 類別
    results = model(frame, classes=[0], conf=conf, verbose=False)
    annotated = results[0].plot()
    count = len(results[0].boxes)

    # 在畫面上用大字顯示人數
    cv2.putText(annotated, f'People: {count}', (10, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)

    return count, annotated, results[0].boxes

# 測試：從新聞台抓一幀計算人數
channel = '城市人潮 (路口直播)'
frame = grab_frame(STREAMS[channel])
count, annotated, boxes = count_people(frame, model)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Detected: {count} people')
axes[1].axis('off')
plt.tight_layout()
plt.show()

# 每個人的詳細資訊
print(f'\n偵測到 {count} 個人：')
for i, box in enumerate(boxes):
    conf = float(box.conf)
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    print(f'  Person {i+1}: confidence={conf:.1%}, position=({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f})')

### A-2: 時間序列人數追蹤

連續監測一段時間，畫出人數隨時間的變化曲線。

In [ ]:
# === 連續追蹤人數（60 秒）===
channel = '城市人潮 (路口直播)'   # 可換頻道
duration = 60              # 追蹤秒數
interval = 3               # 每幾秒偵測一次

cap = cv2.VideoCapture(STREAMS[channel])
start = time.time()
timestamps = []
people_counts = []

print(f'開始追蹤 {channel} 人數（{duration} 秒）...')

while time.time() - start < duration:
    ret, frame = cap.read()
    if not ret:
        break

    count, annotated, _ = count_people(frame, model)
    elapsed = time.time() - start
    timestamps.append(elapsed)
    people_counts.append(count)

    clear_output(wait=True)
    show_frame(annotated, title=f'[{elapsed:.0f}s] {channel} | People: {count}')
    time.sleep(interval)

cap.release()
print(f'追蹤結束！共 {len(timestamps)} 個資料點')

In [ ]:
# === 人數趨勢視覺化 ===
if timestamps:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 折線圖
    axes[0].plot(timestamps, people_counts, 'o-', color='steelblue', linewidth=2, markersize=8)
    axes[0].fill_between(timestamps, people_counts, alpha=0.15, color='steelblue')
    axes[0].set_xlabel('Time (seconds)')
    axes[0].set_ylabel('People Count')
    axes[0].set_title(f'{channel} - People Over Time')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(bottom=0)

    # 統計摘要
    stats_text = f"""Statistics
─────────────────
Duration:  {timestamps[-1]:.0f}s
Samples:   {len(timestamps)}
Min:       {min(people_counts)}
Max:       {max(people_counts)}
Average:   {np.mean(people_counts):.1f}
Std Dev:   {np.std(people_counts):.1f}"""
    axes[1].text(0.1, 0.5, stats_text, fontsize=14, fontfamily='monospace',
                 verticalalignment='center', transform=axes[1].transAxes,
                 bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
    axes[1].axis('off')
    axes[1].set_title('Summary')

    plt.tight_layout()
    plt.show()

### A-3: 區域計數（Zone Counting）

把畫面分成「左半 / 右半」兩個區域，分別計算人數。

```
┌─────────────┬─────────────┐
│  Zone A     │  Zone B     │
│  (Left)     │  (Right)    │
│             │             │
│  👤 👤     │    👤       │
│  Count: 2   │  Count: 1   │
└─────────────┴─────────────┘
```

實際應用：入口 vs 出口、收銀台 vs 賣場

In [ ]:
def count_in_zones(frame, model, conf=0.5):
    """將畫面分成左右兩區，分別計算人數"""
    h, w = frame.shape[:2]
    mid_x = w // 2

    results = model(frame, classes=[0], conf=conf, verbose=False)
    annotated = results[0].plot()

    left_count = 0
    right_count = 0

    for box in results[0].boxes:
        # 用偵測框的中心點判斷在哪個區域
        cx = (box.xyxy[0][0] + box.xyxy[0][2]) / 2
        if cx < mid_x:
            left_count += 1
        else:
            right_count += 1

    # 畫分隔線
    cv2.line(annotated, (mid_x, 0), (mid_x, h), (0, 255, 255), 3)

    # 標註各區人數
    cv2.putText(annotated, f'Zone A: {left_count}', (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 100, 100), 3)
    cv2.putText(annotated, f'Zone B: {right_count}', (mid_x + 10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (100, 100, 255), 3)

    return left_count, right_count, annotated

# 測試
frame = grab_frame(STREAMS['國道1號 高架(車多)'])
left, right, annotated = count_in_zones(frame, model)

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
ax.set_title(f'Zone Counting | Left: {left} | Right: {right} | Total: {left+right}')
ax.axis('off')
plt.show()

---
## Application B: 文字辨識 (OCR)

### 應用場景
- 新聞台跑馬燈文字擷取
- 路標 / 招牌辨識
- 即時字幕擷取

```
新聞畫面
┌────────────────────────────┐
│                            │
│      [主播畫面]            │
│                            │
│ ┌────────────────────────┐ │
│ │ Breaking: AI workshop  │ │ ← OCR 辨識這段文字
│ └────────────────────────┘ │
└────────────────────────────┘
```

我們使用 **EasyOCR** — 支援 80+ 語言的開源 OCR 引擎。

In [ ]:
# === 載入 OCR 引擎 ===
# 第一次執行會下載語言模型（約 100MB），需等 1-2 分鐘
import easyocr

# 載入英文 + 中文辨識
reader = easyocr.Reader(['en', 'ch_tra'], gpu=True)
print('OCR 引擎載入完成！（支援英文 + 繁體中文）')

In [ ]:
# === 對新聞畫面做文字辨識 ===
channel = '城市人潮 (路口直播)'  # 英文新聞台效果最好
frame = grab_frame(STREAMS[channel])

# OCR 辨識
print('正在辨識文字（可能需要 10-20 秒）...')
results = reader.readtext(frame)

# 在影像上標註辨識結果
annotated = frame.copy()
for (bbox, text, conf) in results:
    if conf > 0.3:  # 只顯示信心度 > 30% 的結果
        # 畫框
        pts = np.array(bbox, dtype=np.int32)
        cv2.polylines(annotated, [pts], True, (0, 255, 0), 2)
        # 標註文字
        x, y = int(bbox[0][0]), int(bbox[0][1]) - 5
        cv2.putText(annotated, f'{conf:.0%}', (x, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

# 顯示
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'OCR: {len(results)} text regions')
axes[1].axis('off')
plt.tight_layout()
plt.show()

# 印出辨識到的文字
print(f'\n辨識到 {len(results)} 段文字：')
print('=' * 60)
for bbox, text, conf in sorted(results, key=lambda x: -x[2]):
    if conf > 0.3:
        print(f'  [{conf:.0%}] {text}')

In [ ]:
# === 進階：只擷取畫面下方 1/4 的跑馬燈區域 ===
# 新聞台的跑馬燈通常在畫面最下面

channel = '城市人潮 (路口直播)'
frame = grab_frame(STREAMS[channel])

h, w = frame.shape[:2]
# 只擷取下方 25% 的區域（跑馬燈通常在這裡）
ticker_region = frame[int(h*0.75):h, :]

# OCR
print('辨識跑馬燈區域...')
results = reader.readtext(ticker_region)

# 顯示
annotated = ticker_region.copy()
for (bbox, text, conf) in results:
    if conf > 0.3:
        pts = np.array(bbox, dtype=np.int32)
        cv2.polylines(annotated, [pts], True, (0, 255, 0), 2)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
# 畫出擷取區域
rect = plt.Rectangle((0, h*0.75), w, h*0.25, linewidth=2, edgecolor='r', facecolor='none')
axes[0].add_patch(rect)
axes[0].set_title('Full Frame (red = ticker region)')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title('Ticker Region OCR')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print('\n跑馬燈文字：')
for bbox, text, conf in results:
    if conf > 0.3:
        print(f'  {text}')

---
## Application C: 交通工具偵測與分類

### 應用場景
- 路口車流量統計
- 停車場管理
- 自動駕駛感知

```
YOLO 可辨識的交通工具：
┌────────┬────────┬────────┬────────┐
│bicycle │  car   │  bus   │ truck  │
│ 腳踏車 │  汽車  │  公車  │ 卡車   │
├────────┼────────┼────────┼────────┤
│  motor │airplane│ train  │  boat  │
│ 機車   │ 飛機   │ 火車   │ 船     │
└────────┴────────┴────────┴────────┘
```

In [ ]:
# === 交通工具偵測 ===
vehicle_classes = {
    1: 'bicycle',
    2: 'car',
    3: 'motorcycle',
    4: 'airplane',
    5: 'bus',
    6: 'train',
    7: 'truck',
    8: 'boat',
}

# 旅遊台通常有較多街景
channel = '國道3號'
frame = grab_frame(STREAMS[channel])

# 偵測所有物件
results = model(frame, verbose=False)
annotated = results[0].plot()

# 分類統計
people = 0
vehicles = Counter()
others = Counter()

for box in results[0].boxes:
    cls_id = int(box.cls)
    cls_name = model.names[cls_id]
    if cls_id == 0:
        people += 1
    elif cls_id in vehicle_classes:
        vehicles[cls_name] += 1
    else:
        others[cls_name] += 1

# 顯示結果
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'{channel}')
axes[0].axis('off')

# 圓餅圖
all_counts = {'People': people}
all_counts.update(dict(vehicles))
if others:
    all_counts['Other'] = sum(others.values())

if sum(all_counts.values()) > 0:
    colors = plt.cm.Set2(np.linspace(0, 1, len(all_counts)))
    axes[1].pie(all_counts.values(), labels=all_counts.keys(), colors=colors,
                autopct='%1.0f%%', startangle=90)
    axes[1].set_title('Detection Breakdown')
else:
    axes[1].text(0.5, 0.5, 'No objects detected', ha='center', va='center', fontsize=14)
    axes[1].set_title('Detection Breakdown')

plt.tight_layout()
plt.show()

print(f'行人: {people} | 車輛: {sum(vehicles.values())} ({dict(vehicles)}) | 其他: {sum(others.values())}')

In [ ]:
# === 連續交通監控（30 秒）===
channel = '國道3號'  # 或換其他頻道
duration = 30
interval = 3

cap = cv2.VideoCapture(STREAMS[channel])
start = time.time()
time_data = []
people_data = []
vehicle_data = []

print(f'交通監控中... ({duration}s)')

while time.time() - start < duration:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)
    annotated = results[0].plot()
    elapsed = time.time() - start

    p = sum(1 for b in results[0].boxes if int(b.cls) == 0)
    v = sum(1 for b in results[0].boxes if int(b.cls) in vehicle_classes)

    time_data.append(elapsed)
    people_data.append(p)
    vehicle_data.append(v)

    clear_output(wait=True)
    show_frame(annotated, title=f'[{elapsed:.0f}s] People: {p} | Vehicles: {v}')
    time.sleep(interval)

cap.release()
print('監控結束！')

In [ ]:
# === 交通監控視覺化 ===
if time_data:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(time_data, people_data, 'o-', label='People', color='steelblue', linewidth=2)
    ax.plot(time_data, vehicle_data, 's-', label='Vehicles', color='coral', linewidth=2)
    ax.fill_between(time_data, people_data, alpha=0.1, color='steelblue')
    ax.fill_between(time_data, vehicle_data, alpha=0.1, color='coral')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Count')
    ax.set_title(f'Traffic Monitoring - {channel}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.show()

---
## Application D: YOLO + OCR 合體

最強組合：先用 YOLO 偵測物件，再用 OCR 辨識文字，兩者結合在同一個畫面！

In [ ]:
# === YOLO 物件偵測 + OCR 文字辨識 ===
channel = '國道1號 高架(車多)'
frame = grab_frame(STREAMS[channel])

# Step 1: YOLO 偵測
yolo_results = model(frame, verbose=False)

# Step 2: OCR 辨識
print('OCR 辨識中...')
ocr_results = reader.readtext(frame)

# Step 3: 合併繪製
combined = yolo_results[0].plot()  # YOLO 框（藍/綠色系）

for (bbox, text, conf) in ocr_results:
    if conf > 0.4:
        pts = np.array(bbox, dtype=np.int32)
        cv2.polylines(combined, [pts], True, (0, 255, 255), 2)  # OCR 框（黃色）

# 顯示
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))
ax.set_title(f'{channel} | YOLO (colored boxes) + OCR (yellow boxes)')
ax.axis('off')
plt.tight_layout()
plt.show()

# 統計
obj_counts = Counter(model.names[int(b.cls)] for b in yolo_results[0].boxes)
text_count = sum(1 for _, _, c in ocr_results if c > 0.4)

print(f'\nYOLO 偵測到: {dict(obj_counts)}')
print(f'OCR 辨識到: {text_count} 段文字')
print('\n文字內容:')
for _, text, conf in sorted(ocr_results, key=lambda x: -x[2])[:10]:
    if conf > 0.4:
        print(f'  {text}')

---
## Module 2 完成！

### 你學會了：
- 人數即時計算 + 時間趨勢追蹤
- 區域分區計數
- OCR 文字辨識（整頁 + 跑馬燈區域）
- 交通工具偵測與分類統計
- YOLO + OCR 組合技

### 接下來：
→ 開啟 **03_challenge.ipynb**，進入挑戰賽！